In [ ]:
import os
import glob
import tiktoken
import numpy as np
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sklearn.manifold import TSNE
from langchain_core.messages import SystemMessage, HumanMessage

load_dotenv(override=True)

In [ ]:
model = "gemini-3.5-flash-lite"

db_name = "vector_db"

In [ ]:
embeddings = GoogleGenerativeAIEmbeddings(
model="models/gemini-embedding-2-preview",
google_api_key=os.getenv("GEMINI_API_KEY")
)

vectorstore = Chroma(persist_directory=db_name, embedding_function=embeddings)

## Time to create LLM and retriever

In [ ]:
retriever = vectorstore.as_retriever()
llm = ChatGoogleGenerativeAI(
    temperature=0,
    model=model,
    seed=42,
    google_api_key=os.getenv("GEMINI_API_KEY")
)

In [ ]:
retriever.invoke("Who is Aarav Mehta")

In [ ]:
llm.invoke("Who is Aarav Mehta")

In [ ]:
# Putting llm and encoder together
SYSTEM_PROMPT = """You are a knowledgeable assistant representing a company NovaMind Analytics
If you do not know an answer just say so
Context:{context}"""

In [ ]:
def answer_question(question, history):
    docs = retriever.invoke(question)
    context = "\n".join(doc.page_content for doc in docs)
    system_prompt = SYSTEM_PROMPT.format(context=context)
    response = llm.invoke([SystemMessage(content=system_prompt), HumanMessage(content=question)])
    return response.content

In [ ]:
answer_question("Who is aarav mehta", [])[0]['text']

In [ ]:
# Evaluating RAG

In [ ]:
# import json

# with open(r"C:\training projects\RAG\tests.json", "r", encoding="utf-8") as f:
#     tests = json.load(f)

# score = 0

# for test in tests:
#     question = test["question"]
#     expected = test["expected_answer"]

    

#     answer = answer_question(question, [])[0]['text']

#     success = expected.lower() in answer.lower()

#     if success:
#         score += 1

#     print("=" * 50)
#     print("QUESTION :", question)
#     print("EXPECTED :", expected)
#     print("ANSWER   :", answer)
#     print("PASS     :", success)

# print(f"\nFinal Score: {score}/{len(tests)}")
# print(f"Accuracy: {100 * score / len(tests):.2f}%")

## MRR and recall@k calculation

In [ ]:
total_mrr = 0
total_recall = 0

for test in tests:

    question = test["question"]
    expected = test["expected_answer"].lower()

    docs = retriever.invoke(question)

    # MRR
    reciprocal_rank = 0

    for rank, doc in enumerate(docs, start=1):

        if expected in doc.page_content.lower():
            reciprocal_rank = 1 / rank
            break

    total_mrr += reciprocal_rank

    # Recall@k
    found = any(
        expected in doc.page_content.lower()
        for doc in docs
    )

    total_recall += int(found)

    print("=" * 60)
    print("Question:", question)
    print("Expected:", expected)
    print("MRR:", reciprocal_rank)
    print("Retrieved:", found)

mrr = total_mrr / len(tests)
recall = total_recall / len(tests)

print("\nFINAL RESULTS")
print("MRR =", round(mrr, 4))
print("Recall@k =", round(recall, 4))

## LLM as a judge

In [ ]:
from pydantic import BaseModel

class AnswerEval(BaseModel):
    feedback: str
    accuracy: float
    completeness: float
    relevance: float

judge_llm = llm.with_structured_output(AnswerEval)

In [ ]:
from langchain_core.messages import SystemMessage, HumanMessage
import json

JUDGE_PROMPT = """
You are an expert evaluator for RAG systems.

Compare the generated answer against the reference answer.

Score each metric from 1 to 5.

Scoring rules:

Accuracy:
- 1 = factually wrong
- 3 = partially correct
- 5 = perfectly correct

Completeness:
- 1 = most information missing
- 3 = partially complete
- 5 = contains all expected information

Relevance:
- 1 = off-topic
- 3 = somewhat relevant
- 5 = directly answers the question without unnecessary information

Return ONLY valid JSON:

{
  "feedback": "...",
  "accuracy": 1,
  "completeness": 1,
  "relevance": 1
}
"""

def judge_answer(question, expected_answer, generated_answer):

    prompt = f"""
    Question:
    {question}

    Reference Answer:
    {expected_answer}

    Generated Answer:
    {generated_answer}
    """

    return judge_llm.invoke([
        SystemMessage(content=JUDGE_PROMPT),
        HumanMessage(content=prompt)
    ])

with open("tests.json", "r", encoding="utf-8") as f:
    tests = json.load(f)

results = []

for test in tests:

    question = test["question"]
    expected = test["expected_answer"]

    generated = answer_question(
        question,
        []
    )

    evaluation = judge_answer(
        question,
        expected,
        generated
    )

    result = {
    "question": question,
    "expected": expected,
    "generated": generated,
    "feedback": evaluation.feedback,
    "accuracy": evaluation.accuracy,
    "completeness": evaluation.completeness,
    "relevance": evaluation.relevance,
}

    results.append(result)

    print("=" * 80)
    print("QUESTION:", question)
    print("GENERATED:", generated)
    print("EVALUATION:", evaluation)


avg_accuracy = sum(r["accuracy"] for r in results) / len(results)
avg_completeness = sum(r["completeness"] for r in results) / len(results)
avg_relevance = sum(r["relevance"] for r in results) / len(results)

print("\n" + "=" * 80)
print("FINAL SCORES")
print("=" * 80)

print(f"Accuracy    : {avg_accuracy:.2f}/5")
print(f"Completeness: {avg_completeness:.2f}/5")
print(f"Relevance   : {avg_relevance:.2f}/5")